# 04 — Math & Reductions

**Dataset**: `sklearn.datasets.load_digits` — 1797 handwritten digits, 8×8 grayscale images, 10 classes.  
**Goal**: Normalize pixel values, compute min/max/mean/std of images, argmax, element-wise math  
(exp, log, clamp, sigmoid, ReLU) — side-by-side in NumPy and PyTorch.

In [1]:
# ── Shared Setup ────────────────────────────────────────────────────────────
from sklearn.datasets import load_digits
import numpy as np
import torch
import matplotlib.pyplot as plt

digits = load_digits()

np_images = digits.images.astype(np.float32)   # (1797, 8, 8)
np_labels = digits.target.astype(np.int64)     # (1797,)

pt_images = torch.tensor(np_images)
pt_labels = torch.tensor(np_labels)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'images: {np_images.shape} | pixel range: [{np_images.min()}, {np_images.max()}]')

images: (1797, 8, 8) | pixel range: [0.0, 16.0]


---
## P1 — Global & Per-Image Statistics

Compute the global `min`, `max`, `mean`, and `std` across the entire dataset,  
then per-image mean brightness `(1797,)`.

In [2]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_gmin  = np_images.min()
np_gmax  = np_images.max()
np_gmean = np_images.mean()
np_gstd  = np_images.std()

# Per-image mean brightness: average over spatial dims (axis 1, 2)
np_brightness = np_images.mean(axis=(1, 2))    # (1797,)

print(f'global — min: {np_gmin}, max: {np_gmax}, mean: {np_gmean:.3f}, std: {np_gstd:.3f}')
print(f'brightness shape: {np_brightness.shape}, sample [0..4]: {np_brightness[:5].round(2)}')

global — min: 0.0, max: 16.0, mean: 4.884, std: 6.017
brightness shape: (1797,), sample [0..4]: [4.59 4.89 5.38 4.17 4.03]


#### Drill — `compute_stats`
Practice the core operation before using it in the problem above.

In [3]:
t = torch.tensor([1., 2., 3.])
# DRILL: calc mean and sum
m = t.mean()
s = t.sum()
assert m == 2.0 and s == 6.0

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def compute_stats(imgs):
    """
    Args:  imgs — (1797, 8, 8) tensor

    Returns:
        g_min, g_max, g_mean, g_std — scalar floats
        brightness — (1797,) tensor — per-image mean
    """
    g_min  = ...
    g_max  = ...
    g_mean = ...
    g_std  = ...

    # Per-image: mean over dims (1, 2)  →  (N,)
    # NumPy equivalent: X.mean(axis=(1, 2))
    brightness = ...

    return g_min, g_max, g_mean, g_std, brightness

t_gmin, t_gmax, t_gmean, t_gstd, t_brightness = compute_stats(pt_images)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.isclose(np_gmin,  t_gmin,  atol=1e-5)
assert np.isclose(np_gmax,  t_gmax,  atol=1e-5)
assert np.isclose(np_gmean, t_gmean, atol=1e-4)
assert np.isclose(np_gstd,  t_gstd,  atol=1e-4)
assert t_brightness.shape == (1797,)
assert np.allclose(np_brightness, t_brightness.numpy(), atol=1e-4)
print('P1 assertions passed ✓')

---
## P2 — Min/Max Normalization to [0, 1]

Scale the pixel range from `[0, 16]` to `[0, 1]` for the entire dataset.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_lo  = np_images.min()
np_hi  = np_images.max()
np_01  = (np_images - np_lo) / (np_hi - np_lo)   # (1797, 8, 8) in [0, 1]

print(f'normalised range: [{np_01.min()}, {np_01.max()}]')

#### Drill — `minmax_01`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.tensor([1., 5., 10.])
# DRILL: min and max
mi = t.min()
ma = t.max()
assert mi == 1.0 and ma == 10.0

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def minmax_01(imgs):
    """
    Scale (N, 8, 8) tensor to [0, 1].
    """
    lo = ...
    hi = ...
    normalised = ...
    return normalised

t_01 = minmax_01(pt_images)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_01.shape == (1797, 8, 8)
assert t_01.min() >= -1e-6
assert t_01.max() <=  1 + 1e-6
assert np.allclose(np_01, t_01.numpy(), atol=1e-5)
print('P2 assertions passed ✓')

---
## P3 — Argmax: Predict the Brightest Pixel

For each flat image `(64,)` find the index of the brightest pixel.  
Also find which image in the entire batch is the brightest on average.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_flat = np_images.reshape(len(np_images), -1)         # (1797, 64)
np_argmax_pixel  = np.argmax(np_flat, axis=1)           # (1797,) — brightest pixel index
np_argmax_image  = np.argmax(np_brightness)             # scalar — brightest image index

print(f'brightest pixel per image (first 5): {np_argmax_pixel[:5]}')
print(f'brightest image index: {np_argmax_image}')

#### Drill — `argmax_ops`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.tensor([[10, 20], [30, 5]])
# DRILL: argmax along rows (dim=1)
idx = t.argmax(dim=1)
assert idx[0] == 1 and idx[1] == 0

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def argmax_ops(imgs, brightness):
    """
    Returns:
        t_argmax_pixel : (N,)  — index of brightest pixel per image
        t_argmax_image : int   — index of brightest image overall
    """
    t_flat = imgs.flatten(start_dim=1)              # (N, 64)

    # NumPy equivalent: np.argmax(X, axis=1)
    t_argmax_pixel = ...

    # NumPy equivalent: np.argmax(brightness)
    t_argmax_image = ...

    return t_argmax_pixel, t_argmax_image

t_argmax_pixel, t_argmax_image = argmax_ops(pt_images, t_brightness)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.array_equal(np_argmax_pixel, t_argmax_pixel.numpy())
assert np_argmax_image == t_argmax_image
print('P3 assertions passed ✓')

---
## P4 — Element-wise Math: Sigmoid & ReLU on Pixel Values

Apply common activation functions to the first image (mean-centered first).

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
sample_np = np_images[0] - np_images[0].mean()      # centered, can be negative

np_sigmoid = 1.0 / (1.0 + np.exp(-sample_np))       # sigmoid
np_relu    = np.maximum(sample_np, 0)                # ReLU
np_clamp   = np.clip(sample_np, -5, 5)               # clamp to [-5, 5]
np_abs     = np.abs(sample_np)                       # absolute value

print('sigmoid range:', np_sigmoid.min().round(3), np_sigmoid.max().round(3))
print('ReLU min:', np_relu.min())

#### Drill — `elementwise_math`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.tensor([1., 4., 9.])
# DRILL: square root
sq = torch.sqrt(t)
assert sq[2] == 3.

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def elementwise_math(img_t):
    """
    Apply sigmoid, ReLU, clamp, and abs to a centered image tensor.

    Args:  img_t — (8, 8) tensor (may contain negatives)

    Returns:
        t_sigmoid, t_relu, t_clamp, t_abs — all (8, 8)
    """
    # Sigmoid: 1 / (1 + exp(-x))
    # Can use torch.sigmoid() or manual formula
    t_sigmoid = ...

    # ReLU: max(x, 0)
    # NumPy equivalent: np.maximum(x, 0)
    t_relu = ...

    # Clamp: clip values to [-5, 5]
    # NumPy equivalent: np.clip(x, -5, 5)
    t_clamp = ...

    # Absolute value
    # NumPy equivalent: np.abs(x)
    t_abs = ...

    return t_sigmoid, t_relu, t_clamp, t_abs

sample_t = pt_images[0] - pt_images[0].mean()
t_sigmoid, t_relu, t_clamp, t_abs = elementwise_math(sample_t)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert np.allclose(np_sigmoid, t_sigmoid.numpy(), atol=1e-5)
assert np.allclose(np_relu,    t_relu.numpy(),    atol=1e-5)
assert np.allclose(np_clamp,   t_clamp.numpy(),   atol=1e-5)
assert np.allclose(np_abs,     t_abs.numpy(),     atol=1e-5)
print('P4 assertions passed ✓')

---
## P5 — Topk & Cumulative Sum

Find the top-3 brightest pixels in each flat image, then compute running cumulative brightness.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
np_flat = np_images.reshape(len(np_images), -1)     # (1797, 64)

# Top-3 per image (NumPy doesn't have topk, so we use argsort)
np_top3_idx = np.argsort(np_flat, axis=1)[:, -3:][:, ::-1]   # (1797, 3) descending
np_top3_val = np.take_along_axis(np_flat, np_top3_idx, axis=1)

# Cumulative sum of per-image brightness
np_cumsum = np.cumsum(np_brightness)

print('top3 indices shape:', np_top3_idx.shape)
print('cumsum shape:', np_cumsum.shape)

#### Drill — `topk_and_cumsum`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.tensor([1, 4, 2, 8])
# DRILL: get top 2 and cumulative sum
tvp = torch.topk(t, 2)
cs = t.cumsum(0)
assert tvp.values[0] == 8 and cs[-1] == 15

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def topk_and_cumsum(imgs, brightness):
    """
    Returns:
        topk_vals : (N, 3)  — top-3 pixel values per image
        topk_idx  : (N, 3)  — their indices
        cumsum    : (N,)    — running total of brightness
    """
    t_flat = imgs.flatten(start_dim=1)     # (N, 64)

    # torch.topk — returns namedtuple (values, indices)
    # NumPy equivalent: np.argsort(X, axis=1)[:, -k:]
    topk_result = ...
    topk_vals   = ...
    topk_idx    = ...

    # NumPy equivalent: np.cumsum(x)
    cumsum = ...

    return topk_vals, topk_idx, cumsum

t_top3_val, t_top3_idx, t_cumsum = topk_and_cumsum(pt_images, t_brightness)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_top3_val.shape == (1797, 3)
assert t_top3_idx.shape == (1797, 3)
# topk values should match numpy's sorted values
assert np.allclose(np_top3_val, t_top3_val.numpy(), atol=1e-5)
assert t_cumsum.shape == (1797,)
assert np.allclose(np_cumsum, t_cumsum.numpy(), atol=1e-2)
print('P5 assertions passed ✓')

---
## P6 — Softmax over Class Logits (Preview)

Pretend we have raw logits from a classifier.  
Apply softmax to convert them to probabilities. This is a preview for notebook 06.

In [ ]:
# ── NumPy Reference ──────────────────────────────────────────────────────────
rng = np.random.default_rng(42)
np_logits = rng.standard_normal((5, 10)).astype(np.float32)  # 5 samples, 10 classes

# Numerically stable softmax
np_shifted = np_logits - np_logits.max(axis=1, keepdims=True)
np_exp     = np.exp(np_shifted)
np_softmax = np_exp / np_exp.sum(axis=1, keepdims=True)     # (5, 10)

print('softmax row sums (should be 1):', np_softmax.sum(axis=1).round(4))

#### Drill — `manual_softmax`
Practice the core operation before using it in the problem above.

In [ ]:
t = torch.tensor([0., 0., 0.])
# DRILL: exponential then normalize
exp_t = torch.exp(t)
probs = exp_t / exp_t.sum()
assert probs[0] > 0.33 and probs[0] < 0.34

In [ ]:
# ── PyTorch TODO ─────────────────────────────────────────────────────────────
def manual_softmax(logits_t):
    """
    Implement softmax manually (numerically stable).

    Args:  logits_t — (B, C) tensor of raw scores
    Returns:
        probs : (B, C) tensor — each row sums to 1
    """
    # Step 1: subtract max for numerical stability
    shifted = ...

    # Step 2: exponentiate
    exp_vals = ...

    # Step 3: normalize
    probs = ...

    return probs

t_logits = torch.tensor(np_logits)
t_softmax = manual_softmax(t_logits)

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
assert t_softmax.shape == (5, 10)
assert np.allclose(np_softmax, t_softmax.numpy(), atol=1e-5)
# Each row should sum to 1
assert np.allclose(t_softmax.sum(dim=1).numpy(), 1.0, atol=1e-5)
# Also verify against torch.softmax
assert np.allclose(torch.softmax(t_logits, dim=1).numpy(), t_softmax.numpy(), atol=1e-5)
print('P6 assertions passed ✓')